In [ ]:
import os
from openai import OpenAI
import alpaca_trade_api as tradeapi

In [ ]:
key = os.environ["ALPACA_API_KEY"]
secret_key = os.environ["ALPACA_API_SECRET"]
BASE_URL = "https://paper-api.alpaca.markets/"

api = tradeapi.REST(key, secret_key, BASE_URL, api_version="v2")

In [ ]:
def fetch_portfolio():
    return [
        {
            "symbol": pos.symbol,
            "qty": pos.qty,
            "entry_price": pos.avg_entry_price,
            "current_price": pos.current_price,
            "unrealized_pl": pos.unrealized_pl,
            "side": pos.side,
        }
        for pos in api.list_positions()
    ]


def fetch_open_orders():
    return [
        {
            "symbol": order.symbol,
            "qty": order.qty,
            "limit_price": order.limit_price,
            "side": order.side,
        }
        for order in api.list_orders(status="open")
    ]


def analyze_message(message):
    portfolio_data = fetch_portfolio()
    open_orders = fetch_open_orders()

    pre_prompt = f"""
    You are an AI Portfolio Manager responsible for analyzing my portfolio.
    1. Evaluate risk exposures
    2. Analyze open limit orders
    3. Provide diversification and trade-adjustment insights
    4. Discuss market outlook
    5. Identify risks and suggest management strategies

    Portfolio: {portfolio_data}
    Open orders: {open_orders}

    Priority question: {message}
    """

    client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[{"role": "system", "content": pre_prompt}],
    )
    return response.choices[0].message.content


analysis = analyze_message("How is my portfolio doing?")

In [ ]:
analysis